In [ ]:
from pymmcore_plus import CMMCorePlus, mda
from useq import MDAEvent
import numpy as np
mmc = CMMCorePlus.instance()
mmc.loadSystemConfiguration(r"\Users\Hannah\Desktop\configuration\PVCAM_only.cfg")

mmc.setProperty('Camera-1','Exposure',2.44)
mmc.setProperty('Camera-1','ScanDirection','Up')
mmc.setProperty('Camera-1','ScanMode','Scan Width')
mmc.setProperty('Camera-1','ScanWidth',8)
mmc.setProperty('Camera-1','Port','Dynamic Range')
mmc.setProperty('Camera-1','TriggerMode','Edge Trigger')

In [ ]:
mda_sequence = [
    MDAEvent(metadata={'idx': str(i), 'step_size': r'0.2'})
    for i in range(10)
]
import nidaqmx
task_co = nidaqmx.Task()
task_co.co_channels.add_co_pulse_chan_freq('Dev1/ctr0', name_to_assign_to_channel='trigger_gen', freq=1)

In [ ]:
task_co.timing.cfg_implicit_timing(nidaqmx.constants.AcquisitionType.FINITE,
                                   len(mda_sequence))

In [ ]:
class EngineWithPulse(mda.MDAEngine):
    def __init__(self, mmc: CMMCorePlus, use_hardware_sequencing: bool, pulsetask) -> None:
        super().__init__(mmc)
        self.use_hardware_sequencing = use_hardware_sequencing
        self.task_co = pulsetask
    # def post_sequence_started(self, event: SequencedEvent) -> None:
    #     self.task_co.start()

In [ ]:
#myEngine = EngineWithPulse(mmc, True, task_co)
#mmc.mda.set_engine(myEngine)
mmc.run_mda(iter(mda_sequence))  # add 'output=' parameter to handle output image
task_co.start()

task_co.wait_until_done(100)
mmc.stopSequenceAcquisition()
task_co.stop()
task_co.close()

In [ ]:
mmc.stopSequenceAcquisition()